In [1]:
import torch
import torch.nn as nn
import math

In [2]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.d_k = d_model // num_heads
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        """
        # Q dimension: batch, num_heads, seq_len_q, d_k
        # K dimension: batch, num_heads, seq_len_k, d_k
        # V dimension: batch, num_heads, seq_len_k, d_v
        
        Returns:
            attention: dimesion batch, num_heads, seq_len_q, d_v
            scores: dimension batch, num_heads, seq_len_q, seq_len_k
        """
        d_k = Q.size(-1)
        scores = torch.matmul(Q, K.transpose(-1, -2))
        scores = scores / math.sqrt(d_k)
        if mask != None:
            scores = scores.masked_fill(mask == 0, float('-inf'))

        weights = torch.softmax(scores, dim=-1)
        weights = self.dropout(weights)
        attention = torch.matmul(weights, V)
        return attention, scores

    def forward(self, x, mask=None):
        # (batch, seq_len, d_model) -> (batch, seq_len, d_model)
        Q = self.W_Q(x)
        K = self.W_K(x)
        V = self.W_V(x)

        # (batch, seq_len, d_model) -> (batch, seq_len, num_heads, d_k) -> (batch, num_heads, seq_len, d_k)
        batch = Q.size(0)
        Q = Q.view(batch, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = K.view(batch, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = V.view(batch, -1, self.num_heads, self.d_k).transpose(1, 2)

        # attention dimension: (batch, num_heads, seq_len, d_v)
        attention, _ = self.scaled_dot_product_attention(Q, K, V, mask)
        # (batch, num_heads, seq_len, d_v) -> (batch, seq_len, num_heads, d_v) -> (batch, seq_len, d_model)
        attention = attention.transpose(1, 2).contiguous().view(batch, -1, self.d_model)
        output = self.W_O(attention)
        return output

        
class ForwardFeedNetwork(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        result = self.linear2(
            self.dropout(
                self.relu(
                    self.linear1(x)
                )
            )
        )
        return result

class PostionalEncoding(nn.Module):
    def __init__(self, d_model, max_len, dropout=0.1):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000)) / d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.ffn = ForwardFeedNetwork(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        normed = self.norm1(x)
        attention_output = self.attention(normed, mask)
        x = x + self.dropout1(attention_output)

        normed = self.norm2(x)
        ffn_output = self.ffn(normed)
        x = x + self.dropout2(ffn_output)
        return x

class TransformerEncoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers, num_classes, max_len=512, dropout=0.1):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.positional_encoding = PostionalEncoding(d_model, max_len, dropout)
        self.layers = nn.ModuleList(
            [EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)]
        )
        self.final_norm = nn.LayerNorm(d_model)
        self.classifier = nn.Linear(d_model, num_classes)
        self._init_weights()

    def _init_weights(self):
        nn.init.xavier_uniform_(self.token_embedding.weight)

    def forward(self, input_ids, mask=None):
        # input_ids dimension (batch, seq_len)
        # x dimension (batch, seq_len, d_model)
        x = self.token_embedding(input_ids)
        x = self.positional_encoding(x)
        
        for layer in self.layers:
            x = layer(x)

        x = self.final_norm(x)
        # pooled dimension (batch, d_model)
        pooled = x.mean(dim=1)
        # logits dimension (batch, num_classes)
        logits = self.classifier(pooled)
        return logits


model = TransformerEncoder(10000, 128, 4, 512, 2, 2, max_len=512, dropout=0.1)
print(f"Number of parameteres: {sum(p.numel() for p in model.parameters()):,}")
        

NameError: name 'nn' is not defined

In [24]:
dummy_input = torch.randint(0, 1000, (4, 32))
output = model(dummy_input)    
print(f"Output shape: {output.shape}")
    

Output shape: torch.Size([4, 2])


In [1]:
def test_shapes():
    batch, seq_len, d_model, num_heads = 2, 10, 128, 4
          MultiHeadAttention 
    mha = MultiHeadAttention(d_model, num_heads)
    x = torch.randn(batch, seq_len, d_model)
    output = mha(x)
    assert output.shape == (batch, seq_len, d_model), f"MultiHeadAttention shape wrong: {output.shape}"
    print(f"MultiHeadAttention shape correct")

    ffn = ForwardFeedNetwork(d_model, 4*d_model)
    x = torch.randn(batch, seq_len, d_model)
    output = ffn(x)
    assert output.shape == (batch, seq_len, d_model), f"ForwardFeedNetwork shape wrong: {output.shape}"
    print(f"ForwardFeedNetwork shape correct")

    # d_model, num_heads, d_ff, dropout=0.1
    encoder_layer = EncoderLayer(d_model, num_heads, 4*d_model)
    x = torch.randn(batch, seq_len, d_model)
    output = encoder_layer(x)
    assert output.shape == (batch, seq_len, d_model), f"Encoder Layer shape wrong: {output.shape}"
    print(f"Encoder Layer shape correct")

    # vocab_size, d_model, num_heads, d_ff, num_layers, num_classes, max_len=512, dropout=0.1
    model = TransformerEncoder(10000, d_model, num_heads, 4*d_model, 2, 2, max_len=512, dropout=0.1)
    input_ids = torch.randint(0, 10000, (batch, seq_len))
    output = encoder_layer(input_ids)
    assert output.shape == (batch, 2), f"Transformer Encoder shape wrong: {output.shape}"
    print(f"Transformer Encoder shape correct")
    
test_shapes()

NameError: name 'MultiHeadAttention' is not defined